# Phase 5 — Dataset Collection
## Educational Rewriter GPT

This notebook collects the source passages that will be used to train the rewriter model.

The goal is **150 confusing educational passages** across three sources:
- **Wikipedia** (80) — technical articles across diverse domains
- **arXiv** (50) — academic paper abstracts, dense and jargon-heavy


These passages will be fed to the Claude API in the next notebook to generate 900 rewrite pairs (150 passages × 6 modes).

---

### Setup

In [7]:
import requests
import json
import time
import random
import re
from pathlib import Path

# Create data directory
Path("data").mkdir(exist_ok=True)
Path("data/raw").mkdir(exist_ok=True)

print("Setup complete!")
print("Target: 150 passages (80 Wikipedia + 50 arXiv)")

Setup complete!
Target: 150 passages (80 Wikipedia + 50 arXiv)


---
### Wikipedia passages

Targeting **80 passages** across computer science, biology, physics, mathematics, economics, chemistry, and medicine.

In [8]:
import urllib.request
import urllib.parse
import json
import re
import time
import random

def get_wikipedia_section(title):
    """
    Fetch a Wikipedia article's wikitext.
    Must include User-Agent or Wikipedia returns 403.
    """
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "parse",
        "page": title,
        "prop": "wikitext",
        "format": "json"
    }
    
    query = urllib.parse.urlencode(params)
    full_url = f"{url}?{query}"
    
    # Wikipedia requires a User-Agent header
    headers = {
        "User-Agent": "Mozilla/5.0 (educational research project; anisharay08@gmail.com)"
    }
    
    try:
        req = urllib.request.Request(full_url, headers=headers)
        with urllib.request.urlopen(req, timeout=10) as response:
            data = json.loads(response.read().decode())
        
        if "error" in data:
            return None
            
        wikitext = data.get("parse", {}).get("wikitext", {}).get("*", "")
        return wikitext
    
    except Exception as e:
        print(f"Error fetching {title}: {e}")
        return None

def clean_wikitext(text):
    """Remove wiki markup to get clean text."""
    text = re.sub(r'\{\{[^}]*\}\}', '', text)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]*)\]\]', r'\1', text)
    text = re.sub(r'\[https?://[^\s\]]+\s*([^\]]*)\]', r'\1', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'={2,}[^=]+={2,}', '', text)
    text = re.sub(r'\[\[(?:File|Image):[^\]]*\]\]', '', text)
    text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()
    return text

def extract_dense_paragraphs(wikitext, min_len=100, max_len=600):
    """Extract dense, technical paragraphs."""
    if not wikitext:
        return []
    
    clean = clean_wikitext(wikitext)
    paragraphs = clean.split('\n\n')
    
    good_passages = []
    for para in paragraphs:
        para = para.strip()
        
        if len(para) < min_len or len(para) > max_len:
            continue
        
        lines = para.split('\n')
        if sum(1 for l in lines if l.strip().startswith('*')) > len(lines) * 0.5:
            continue
        
        digit_ratio = sum(c.isdigit() for c in para) / len(para)
        if digit_ratio > 0.15:
            continue
        
        sentences = re.split(r'[.!?]+', para)
        avg_sent_len = sum(len(s.split()) for s in sentences) / max(len(sentences), 1)
        if avg_sent_len < 8:
            continue
        
        good_passages.append(para)
    
    return good_passages

# Topics list (same as before)
wikipedia_topics = [
    "Artificial neural network",
    "Backpropagation",
    "Convolutional neural network",
    "Support vector machine",
    "Gradient descent",
    "Reinforcement learning",
    "Natural language processing",
    "Transformer (machine learning model)",
    "Recurrent neural network",
    "Attention (machine learning)",
    "DNA replication",
    "Protein folding",
    "Mitochondrion",
    "CRISPR",
    "Photosynthesis",
    "Cell membrane",
    "Enzyme",
    "Immune system",
    "Quantum mechanics",
    "Special relativity",
    "Thermodynamics",
    "Maxwell's equations",
    "Wave–particle duality",
    "Black hole",
    "Entropy",
    "Calculus",
    "Linear algebra",
    "Fourier transform",
    "Differential equation",
    "Probability theory",
    "Graph theory",
    "Number theory",
    "Game theory",
    "Supply and demand",
    "Monetary policy",
    "Keynesian economics",
    "Market failure",
    "Chemical bond",
    "Organic chemistry",
    "Electrochemistry",
    "Thermochemistry",
    "Periodic table",
    "Pharmacokinetics",
    "Inflammation",
    "Autoimmune disease",
    "Pharmacodynamics",
    "Neurotransmitter",
]

print(f"Fetching Wikipedia passages from {len(wikipedia_topics)} topics...")
print("This will take a few minutes...\n")

wikipedia_passages = []

for i, topic in enumerate(wikipedia_topics):
    print(f"[{i+1}/{len(wikipedia_topics)}] Fetching: {topic}")
    
    wikitext = get_wikipedia_section(topic)
    passages = extract_dense_paragraphs(wikitext)
    
    selected = passages[:2]
    
    for passage in selected:
        wikipedia_passages.append({
            "text": passage,
            "source": "wikipedia",
            "topic": topic,
            "length": len(passage.split()),
            "length_type": "paragraph" if len(passage.split()) < 100 else "section"
        })
    
    time.sleep(0.5)

print(f"\nCollected {len(wikipedia_passages)} Wikipedia passages")
print(f"Target: 80. {'✅ Good!' if len(wikipedia_passages) >= 60 else '⚠️ Need more topics'}")

Fetching Wikipedia passages from 47 topics...
This will take a few minutes...

[1/47] Fetching: Artificial neural network
[2/47] Fetching: Backpropagation
[3/47] Fetching: Convolutional neural network
[4/47] Fetching: Support vector machine
[5/47] Fetching: Gradient descent
[6/47] Fetching: Reinforcement learning
[7/47] Fetching: Natural language processing
[8/47] Fetching: Transformer (machine learning model)
[9/47] Fetching: Recurrent neural network
[10/47] Fetching: Attention (machine learning)
[11/47] Fetching: DNA replication
[12/47] Fetching: Protein folding
[13/47] Fetching: Mitochondrion
[14/47] Fetching: CRISPR
[15/47] Fetching: Photosynthesis
[16/47] Fetching: Cell membrane
[17/47] Fetching: Enzyme
[18/47] Fetching: Immune system
[19/47] Fetching: Quantum mechanics
[20/47] Fetching: Special relativity
[21/47] Fetching: Thermodynamics
[22/47] Fetching: Maxwell's equations
[23/47] Fetching: Wave–particle duality
[24/47] Fetching: Black hole
[25/47] Fetching: Entropy
[26/47] Fet

---
### arXiv abstracts

Targeting **50 abstracts** across ML, NLP, biology, quantum computing, economics, and statistics.

In [9]:
def fetch_arxiv_abstracts(query, max_results=10):
    """
    Fetch abstracts from arXiv API.
    Returns list of abstract strings.
    """
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": query,
        "start": 0,
        "max_results": max_results,
        "sortBy": "relevance",
        "sortOrder": "descending"
    }
    
    query_str = urllib.parse.urlencode(params)
    url = f"{base_url}?{query_str}"
    
    try:
        with urllib.request.urlopen(url, timeout=15) as response:
            content = response.read().decode()
        
        # Extract abstracts using simple regex
        abstracts = re.findall(r'<summary>(.*?)</summary>', content, re.DOTALL)
        titles = re.findall(r'<title>(.*?)</title>', content, re.DOTALL)
        
        # Skip the first title (it's the feed title)
        titles = titles[1:] if len(titles) > 1 else titles
        
        results = []
        for i, abstract in enumerate(abstracts):
            abstract = abstract.strip()
            abstract = re.sub(r'\s+', ' ', abstract)
            
            # Filter: good length for our purposes
            if 100 <= len(abstract.split()) <= 300:
                results.append({
                    "text": abstract,
                    "title": titles[i] if i < len(titles) else "Unknown",
                })
        
        return results
    
    except Exception as e:
        print(f"Error fetching arXiv: {e}")
        return []

# arXiv search queries — technical, diverse
arxiv_queries = [
    "cat:cs.LG deep learning neural networks",
    "cat:cs.CL natural language processing transformers",
    "cat:cs.AI reinforcement learning",
    "cat:q-bio.BM protein structure prediction",
    "cat:cond-mat.mes-hall quantum computing",
    "cat:econ.GN macroeconomic policy",
    "cat:cs.RO robotics autonomous systems",
    "cat:stat.ML statistical machine learning",
    "cat:cs.CV computer vision object detection",
    "cat:math.ST statistical theory",
    # Extra queries to reach 70
    "cat:cs.NE evolutionary computation",
    "cat:q-bio.NC computational neuroscience",
    "cat:physics.ed-ph physics education",
    "cat:cs.DB database systems query optimization",
]

print("Fetching arXiv abstracts...")
print("This will take a minute...\n")

arxiv_passages = []

for i, query in enumerate(arxiv_queries):
    print(f"[{i+1}/{len(arxiv_queries)}] Query: {query[:50]}...")
    
    results = fetch_arxiv_abstracts(query, max_results=10)
    
    for result in results[:8]:
        arxiv_passages.append({
            "text": result["text"],
            "source": "arxiv",
            "topic": result["title"][:50],
            "length": len(result["text"].split()),
            "length_type": "paragraph"
        })
    
    time.sleep(1)  # Be polite to arXiv API

print(f"\nCollected {len(arxiv_passages)} arXiv passages")
print(f"Target: 50. {'✅ Good!' if len(arxiv_passages) >= 40 else '⚠️ Need more queries'}")

Fetching arXiv abstracts...
This will take a minute...

[1/14] Query: cat:cs.LG deep learning neural networks...
[2/14] Query: cat:cs.CL natural language processing transformers...
[3/14] Query: cat:cs.AI reinforcement learning...
[4/14] Query: cat:q-bio.BM protein structure prediction...
[5/14] Query: cat:cond-mat.mes-hall quantum computing...
[6/14] Query: cat:econ.GN macroeconomic policy...
[7/14] Query: cat:cs.RO robotics autonomous systems...
[8/14] Query: cat:stat.ML statistical machine learning...
[9/14] Query: cat:cs.CV computer vision object detection...
[10/14] Query: cat:math.ST statistical theory...
[11/14] Query: cat:cs.NE evolutionary computation...
[12/14] Query: cat:q-bio.NC computational neuroscience...
[13/14] Query: cat:physics.ed-ph physics education...
[14/14] Query: cat:cs.DB database systems query optimization...

Collected 111 arXiv passages
Target: 50. ✅ Good!


---
### Combining and saving

Merging all two sources into a single dataset file with consistent schema. Each passage gets a unique ID, source label, topic, word count, and length type (sentence / paragraph / section).

In [10]:
# Combine all passages
all_passages = wikipedia_passages + arxiv_passages 

# Add unique IDs
for i, passage in enumerate(all_passages):
    passage["id"] = f"passage_{i:03d}"

# Stats
print("="*60)
print("DATASET STATISTICS")
print("="*60)
print(f"\nTotal passages collected: {len(all_passages)}")
print(f"  Wikipedia: {len(wikipedia_passages)}")
print(f"  arXiv:     {len(arxiv_passages)}")

print(f"\nLength distribution:")
lengths = [p["length"] for p in all_passages]
print(f"  Min words:  {min(lengths)}")
print(f"  Max words:  {max(lengths)}")
print(f"  Avg words:  {sum(lengths)/len(lengths):.0f}")

length_types = {}
for p in all_passages:
    lt = p.get("length_type", "unknown")
    length_types[lt] = length_types.get(lt, 0) + 1
print(f"\nLength types:")
for lt, count in length_types.items():
    print(f"  {lt}: {count}")

print(f"\nSource distribution:")
sources = {}
for p in all_passages:
    s = p["source"]
    sources[s] = sources.get(s, 0) + 1
for s, count in sources.items():
    print(f"  {s}: {count}")

# Save to JSON
output_path = "data/raw/passages.json"
with open(output_path, "w") as f:
    json.dump(all_passages, f, indent=2, ensure_ascii=False)

print(f"\n✅ Saved {len(all_passages)} passages to {output_path}")
print("\nSample passages:")
for p in random.sample(all_passages, min(3, len(all_passages))):
    print(f"\n[{p['source']} — {p['topic']}]")
    print(f"  {p['text'][:150]}...")

DATASET STATISTICS

Total passages collected: 201
  Wikipedia: 90
  arXiv:     111

Length distribution:
  Min words:  12
  Max words:  284
  Avg words:  125

Length types:
  paragraph: 201

Source distribution:
  wikipedia: 90
  arxiv: 111

✅ Saved 201 passages to data/raw/passages.json

Sample passages:

[arxiv — A Comprehensive Review of State-of-The-Art Methods]
  Java Code Generation consists in generating automatically Java code from a Natural Language Text. This NLP task helps in increasing programmers' produ...

[wikipedia — Black hole]
  Quantum field theory in curved spacetime predicts that event horizons emit Hawking radiation, with its rate of emission being inversely proportional t...

[arxiv — Query Optimization Techniques In Graph Databases]
  Graph databases (GDB) have recently been arisen to overcome the limits of traditional databases for storing and managing data with graph-like structur...


---
### Quality check

Before moving to generation, checking for:
- Duplicates
- Passages that are too short or too long
- Domain diversity
- Overall readiness for the Claude API generation step

In [11]:
# Spot check before moving to generation
print("="*60)
print("QUALITY CHECK")
print("="*60)

# Check for duplicates
texts = [p["text"] for p in all_passages]
unique_texts = set(texts)
print(f"\nDuplicates: {len(texts) - len(unique_texts)}")

# Check for very short passages
short = [p for p in all_passages if p["length"] < 30]
print(f"Very short passages (<30 words): {len(short)}")
if short:
    print("  Consider removing these:")
    for p in short:
        print(f"  [{p['id']}] {p['text'][:100]}...")

# Check for very long passages
long = [p for p in all_passages if p["length"] > 500]
print(f"Very long passages (>500 words): {len(long)}")
if long:
    print("  Consider splitting these")

# Check domain diversity
topics = [p["topic"] for p in all_passages]
print(f"\nUnique topics: {len(set(topics))}")

print(f"\n{'✅ Ready for generation!' if len(all_passages) >= 100 else '⚠️ Need more passages before generating'}")
print(f"Collected {len(all_passages)}/150 target passages")

QUALITY CHECK

Duplicates: 0
Very short passages (<30 words): 9
  Consider removing these:
  [passage_000] In machine learning, '''backpropagation''' is a gradient computation method commonly used for traini...
  [passage_011] Major processing tasks in an NLP system include: speech recognition, text classification, natural la...
  [passage_017] DNA replication, like all biological polymerization processes, proceeds in three enzymatically catal...
  [passage_062] thumb|class=skin-invert-image|Supply and demand curves with [[economic equilibrium of price and quan...
  [passage_065] How best to conduct monetary policy is an active and debated research area, drawing on fields like m...
  [passage_071] The atoms in molecules, crystals, metals and other forms of matter are held together by chemical bon...
  [passage_072] Organic chemistry is typically taught at the college or university level. It is considered a very ch...
  [passage_073] After Wöhler, Justus von Liebig worked on the organiz

In [12]:
import random

print("="*60)
print("CLEANING DATASET")
print("="*60)

# Step 1: Remove short passages (<30 words)
before = len(all_passages)
all_passages = [p for p in all_passages if p["length"] >= 30]

# Also remove passages with wiki markup still in them
all_passages = [p for p in all_passages if "[[" not in p["text"] and "{{" not in p["text"] and "thumb|" not in p["text"]]

after_clean = len(all_passages)
print(f"\nRemoved {before - after_clean} bad passages")
print(f"Remaining: {after_clean}")

# Step 2: Trim to 150, keeping source balance
random.seed(42)

wikipedia_clean = [p for p in all_passages if p["source"] == "wikipedia"]
arxiv_clean = [p for p in all_passages if p["source"] == "arxiv"]

print(f"\nBefore trim:")
print(f"  Wikipedia: {len(wikipedia_clean)}")
print(f"  arXiv:     {len(arxiv_clean)}")

# Sample to get balanced 150
wikipedia_final = random.sample(wikipedia_clean, min(75, len(wikipedia_clean)))
arxiv_final = random.sample(arxiv_clean, min(75, len(arxiv_clean)))

all_passages_final = wikipedia_final + arxiv_final
random.shuffle(all_passages_final)

# Re-assign IDs
for i, p in enumerate(all_passages_final):
    p["id"] = f"passage_{i:03d}"

print(f"\nAfter trim (balanced 75/75):")
print(f"  Wikipedia: {len(wikipedia_final)}")
print(f"  arXiv:     {len(arxiv_final)}")
print(f"  Total:     {len(all_passages_final)}")

# Save cleaned version
output_path = "data/raw/passages_clean.json"
with open(output_path, "w") as f:
    json.dump(all_passages_final, f, indent=2, ensure_ascii=False)

print(f"\n✅ Saved {len(all_passages_final)} clean passages to {output_path}")

# Final quality check
lengths = [p["length"] for p in all_passages_final]
print(f"\nFinal length distribution:")
print(f"  Min words: {min(lengths)}")
print(f"  Max words: {max(lengths)}")
print(f"  Avg words: {sum(lengths)/len(lengths):.0f}")
print(f"\n✅ Dataset ready for generation!")

CLEANING DATASET

Removed 24 bad passages
Remaining: 177

Before trim:
  Wikipedia: 66
  arXiv:     111

After trim (balanced 75/75):
  Wikipedia: 66
  arXiv:     75
  Total:     141

✅ Saved 141 clean passages to data/raw/passages_clean.json

Final length distribution:
  Min words: 30
  Max words: 263
  Avg words: 121

✅ Dataset ready for generation!


---
## Results

Collected and cleaned **141 passages** ready for generation.

| Source | Count |
|--------|-------|
| Wikipedia | 66 |
| arXiv | 75 |
| **Total** | **141** |

**141 passages × 6 modes = 846 training examples** — within our 700-1000 target.

Removed 24 passages that were too short, contained leftover wiki markup, or were low quality.

Saved to `data/raw/passages_clean.json`.

---
## Next steps

**Next notebook:** `02_generate_rewrites.ipynb`
- Load `passages_clean.json`
- Write Claude API prompt templates for each of the 6 modes
- Generate 846 (input, mode, output) triplets
- Validate output quality on a sample
- Save as HuggingFace Dataset format

---
*Phase 5 — Educational Rewriter GPT | Dataset Collection*